# Sense & Stop: Member 3 PDE model, run and test in Colab
This notebook runs and tests the PDE model (excitable + Fisher-KPP), calibrated to the S1 rat CSD features.
**Scope:** computational study; rat-derived timing; not clinical, not human data.

**Steps:** 1) upload the zip  2) install  3) run the 21 tests  4) run the model and plot  5) 2D  6) recalibrate.

## 1. Get the code into Colab
**Option A (recommended now):** run the cell and choose `member3_pde_model_v0.2.zip` when asked.

**Option B (after the pull request is merged):** skip Option A and use the `git clone` cell below it.

In [ ]:
# --- Option A: upload the zip ---
import os, sys, zipfile
from google.colab import files
up = files.upload()                       # choose member3_pde_model_v0.2.zip
zipfile.ZipFile(list(up.keys())[0]).extractall(".")
os.chdir("sense-and-stop")
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())
print(sorted(os.listdir(".")))

In [ ]:
# --- Option B: clone from GitHub (use ONLY if you did not run Option A) ---
# import os, sys, subprocess
# subprocess.run(["git", "clone", "https://github.com/Jana-Hazem/sense-and-stop.git"])
# os.chdir("sense-and-stop")
# subprocess.run(["git", "checkout", "member3/pde-model-calibrated"])   # your branch name, or stay on main after merge
# sys.path.insert(0, os.getcwd())
# print(sorted(os.listdir(".")))

## 2. Install and check versions
The tests need **numpy 2.0 or newer** (they use `np.trapezoid`).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "pyyaml", "scipy", "matplotlib"])
import numpy, scipy, yaml, matplotlib
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
assert int(numpy.__version__.split(".")[0]) >= 2, "Upgrade numpy: pip install -U numpy, then Runtime > Restart session"


## 3. Run the 21 tests (about 20 to 60 seconds)
Expected last line: `21 passed`.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-1500:])

## 4. Run the excitable model (1D) and look at it
The pulse should last about **42.8 s** at a point, travel at about **2.6 mm/min**, and be about **1.85 mm** wide.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.model import (simulate, load_config, with_resolution, pulse_metrics, pulse_speed,
                       measure_front_speed, calibrate_excitable, save_result)

cfg = load_config("configs/model_default.yaml")          # excitable, 1D, calibrated
res = simulate(cfg)
print("u shape [t, x]:", res["u"].shape, "| solver:", res["meta"]["solver"], "| dt =", res["meta"]["dt_internal_s"], "s")

m = pulse_metrics(res, 15.0)
c = pulse_speed(res, 10.0, 20.0)
print(f"FWHM      : {m['fwhm_s']:.1f} s     (S1 data: 42.8 s)")
print(f"rise 10%  : {m['rise_10pct_s']:.1f} s     (S1 data: 35.2 s)")
print(f"decay 10% : {m['decay_10pct_s']:.1f} s     (S1 data: 40.7 s; known mismatch)")
print(f"speed     : {c*60:.2f} mm/min (literature 2.3 to 2.9)")
print(f"width     : {c*m['fwhm_s']:.2f} mm   (derived 1.7 to 2.1)")
print(f"u range   : {res['u'].min():.2f} to {res['u'].max():.2f}  (note: dips below 0 after the pulse)")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
im = ax[0].imshow(res["u"], aspect="auto", origin="lower", cmap="viridis", vmin=-0.3, vmax=1,
                  extent=[res["x"][0], res["x"][-1], res["t"][0], res["t"][-1]])
ax[0].set_title("u(x,t): space-time map"); ax[0].set_xlabel("x (mm)"); ax[0].set_ylabel("t (s)"); plt.colorbar(im, ax=ax[0])
j = np.argmin(abs(res["x"] - 15)); ax[1].plot(res["t"], res["u"][:, j]); ax[1].axhline(0, c="k", lw=.5)
ax[1].set_title("One electrode (x = 15 mm)"); ax[1].set_xlabel("t (s)"); ax[1].set_ylabel("u")
i = np.searchsorted(res["t"], 300); ax[2].plot(res["x"], res["u"][i]); ax[2].axhline(0, c="k", lw=.5)
ax[2].set_title("Snapshot at t = 300 s"); ax[2].set_xlabel("x (mm)"); ax[2].set_ylabel("u")
plt.tight_layout(); plt.show()

## 5. Compare with the Fisher-KPP baseline
Fisher-KPP advances and **stays elevated** (no recovery); the excitable model is a pulse that recovers.

In [ ]:
kc = load_config("configs/model_default.yaml"); kc["model"]["type"] = "fisher_kpp"
kpp = simulate(kc)
print("KPP u range:", round(kpp["u"].min(), 4), "to", round(kpp["u"].max(), 4))
print("KPP front speed (t=300..500 s): %.2f mm/min  (limit 2*sqrt(D r) = %.2f mm/min; approaches from below)" % (
      60*measure_front_speed(kpp, t_window=(300, 500)), 60*2*np.sqrt(0.01296*0.036)))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, r_, name in [(ax[0], kpp, "Fisher-KPP"), (ax[1], res, "Excitable")]:
    for ti, col in zip([100, 250, 400], ["C0", "C1", "C2"]):
        k = int(ti / 2); a.plot(r_["x"], r_["u"][k], col, label=f"t={ti}s")
    a.axhline(0, c="k", lw=.5); a.set_title(name + ": profiles"); a.set_xlabel("x (mm)"); a.legend()
plt.tight_layout(); plt.show()

## 6. Avoiding the inverse crime (what Members 4 and 5 will do)
Truth on a fine grid with RK4; an independent coarse solve with scipy `solve_ivp`.

In [ ]:
truth = simulate(with_resolution(cfg, dx=0.05, dt=0.05, method="rk4"))
inv   = simulate(with_resolution(cfg, dx=0.1, method="solve_ivp"))
ref = truth["u"][:, ::2]
print("truth nodes:", truth["u"].shape[1], "| coarse nodes:", inv["u"].shape[1], "| nodes coincide:", np.allclose(truth["x"][::2], inv["x"]))
print("noise-free numerical gap: relative RMS = %.3f  (max abs %.3f)" % (np.sqrt(((ref-inv['u'])**2).mean())/np.sqrt((ref**2).mean()), np.abs(ref-inv['u']).max()))

## 7. 2D run (about 30 to 90 seconds on Colab; uses about 260 MB)
Default patch: 30 x 20 mm, grid 0.1 mm. Array order is `u[t, y, x]`.

In [ ]:
import time
c2 = with_resolution(cfg); c2["model"]["dimension"] = 2
t0 = time.time(); r2 = simulate(c2); print("2D done in %.0f s" % (time.time() - t0), "| u shape [t, y, x]:", r2["u"].shape)
m2 = pulse_metrics(r2, 15.0); print("2D FWHM at x=15 mm: %.1f s | speed %.2f mm/min" % (m2["fwhm_s"], 60*pulse_speed(r2, 10, 20)))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, ti in zip(ax, [150, 300, 450]):
    k = int(ti / 2)
    im = a.imshow(r2["u"][k], origin="lower", cmap="viridis", vmin=-0.3, vmax=1,
                  extent=[r2["x"][0], r2["x"][-1], r2["y"][0], r2["y"][-1]])
    a.set_title(f"t = {ti} s"); a.set_xlabel("x (mm)"); a.set_ylabel("y (mm)")
plt.colorbar(im, ax=ax, shrink=0.8); plt.show()

## 8. Recalibrate to different targets
When Member 2 updates the numbers, change them here. `calibrate_excitable` computes D and k for you.

In [ ]:
FWHM_S = 42.8                 # pulse duration at a point (s)   <- edit
SPEED_MM_MIN = 2.6            # front speed (mm/min)             <- edit (literature range 2.0 to 9.0)

new, info = calibrate_excitable(cfg, fwhm_s=FWHM_S, speed_mm_per_min=SPEED_MM_MIN)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in info.items() if k != "pulse_shape_dimensionless"})
new = with_resolution(new, dx=min(0.1, info["suggested_max_dx_mm"]))
new["time"]["T"] = min(1200.0, 3 + (27.0 / (SPEED_MM_MIN/60)) * 0.95 + 20)   # long enough to cross about 27 mm
chk = simulate(new)
mm = pulse_metrics(chk, 15.0)
print("check by simulation: FWHM %.1f s | speed %.2f mm/min | width %.2f mm" % (
      mm["fwhm_s"], 60*pulse_speed(chk, 10, 20), pulse_speed(chk, 10, 20)*mm["fwhm_s"]))

## 9. Save a result and download it
`save_result` writes the arrays plus metadata to one `.npz` file.

In [ ]:
os.makedirs("outputs", exist_ok=True)
save_result(res, "outputs/run_excitable_1d.npz")
from src.model import load_result
back = load_result("outputs/run_excitable_1d.npz"); print("reloaded u:", back["u"].shape, "| model:", back["params"]["model_type"])
# files.download("outputs/run_excitable_1d.npz")      # uncomment to download

## If something goes wrong
| Problem | Fix |
|---|---|
| `ModuleNotFoundError: src` | Run the upload cell first, and make sure the working folder is `sense-and-stop` |
| `np.trapezoid` error | numpy older than 2.0: `pip install -U numpy`, then **Runtime > Restart session** |
| A test fails after you changed parameters | The excitable pulse is fragile in `a`, `eps`, `gamma`; restore the defaults and use `calibrate_excitable` |
| 2D cell is slow or runs out of memory | Increase `dx`/`dy` to 0.2 for a quick look, or shorten `T` |